In [17]:
# 1. Cleaning data tables for next steps in the pipeline of the Data Processing and Analytics Project in master DISS 2026/2026: The goal here is to move from bronze (Parquet) layer to  silver (clean Parquet)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum, avg, max, count, date_format, to_timestamp

# Spark has only 1 GB by default, so we give it 4 GB to avoid a crash with this size of data
spark = SparkSession.builder \
    .appName("NYC Subway - Part 1 - clean") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

BRONZE = "/home/jovyan/project/data/bronze"
SILVER = "/home/jovyan/project/data/silver"
RESULTS = "/home/jovyan/project/results"

In [18]:
# 2. Ridership  1st Rule R1: keep the subway only (problem #4)
rides = spark.read.parquet(BRONZE + "/ridership")
r0 = rides.count()
rides = rides.where("transit_mode = 'subway'")
r1 = rides.count()
print("R1:", r0, "->", r1)

R1: 29375687 -> 29145131


In [19]:
# 3. Ridership 2nd Rule R2: missing or impossible values to drop empty or negative values 
rides = rides.where("transit_timestamp is not null and station_complex_id is not null and ridership >= 0")
r2 = rides.count()
print("R2:", r1, "->", r2)

R2: 29145131 -> 29145131


In [20]:
# 4. Ridership 3rd rule R3: keep the columns we need, with the right type that means that we keep 5 columns and fix their types (station ID is a text → switched to a number, ridership is a decimal nnumber  → convert into a whole number)
rides = rides.select(
    col("transit_timestamp"),
    col("station_complex_id").cast("int").alias("complex_id"),
    col("payment_method"),
    col("fare_class_category"),
    col("ridership").cast("int").alias("ridership"),
)
rides = rides.where("complex_id is not null")
r3 = rides.count()
print("R3:", r2, "->", r3)

R3: 29145131 -> 29145131


In [21]:
# 5. Ridership  4th rule R4: the hour that does not exist in New York
# 9 March 2025: clocks jump from 01:59 to 03:00, so 02:00-02:59 never happened so we remve that hour that never existed
rides = rides.where("transit_timestamp != '2025-03-09 02:00:00'")
r4 = rides.count()
print("R4:", r3, "->", r4)

R4: 29145131 -> 29143881


In [22]:
# 6. Ridership 5th rule  R5: We delete the duplicate rows
rides = rides.distinct()
r5 = rides.count()
print("R5:", r4, "->", r5)

R5: 29143881 -> 29143881


In [23]:
# 7. Ridership 6th rule R6: one row per station and hour = sum of the fare types (problem #3)
station_hour = rides.groupBy("complex_id", "transit_timestamp").agg(sum("ridership").alias("riders"))
station_hour.write.mode("overwrite").parquet(SILVER + "/station_hour")
r6 = spark.read.parquet(SILVER + "/station_hour").count()
print("R6:", r5, "->", r6)

R6: 29143881 -> 3620146


In [24]:
# 8. Stations 1st rule S1: borough codes -> full names (problem #8)
stations = spark.read.parquet(BRONZE + "/stations")
s0 = stations.count()
stations = stations.withColumn("borough",
    when(col("borough") == "M", "Manhattan")
    .when(col("borough") == "Bk", "Brooklyn")
    .when(col("borough") == "Q", "Queens")
    .when(col("borough") == "Bx", "Bronx")
    .when(col("borough") == "SI", "Staten Island"))
stations.groupBy("borough").count().show()

+-------------+-----+
|      borough|count|
+-------------+-----+
|       Queens|   83|
|     Brooklyn|  169|
|Staten Island|   21|
|    Manhattan|  153|
|        Bronx|   70|
+-------------+-----+



In [25]:
# 9. Stations 2nd Rule S2: one row per station complex (problem #7)
complexes = stations.groupBy("complex_id").agg(
    max("stop_name").alias("complex_name"),
    max("borough").alias("borough"),
    count("*").alias("n_stops"),
)
complexes.write.mode("overwrite").parquet(SILVER + "/complexes")
s1 = spark.read.parquet(SILVER + "/complexes").count()
print("S2:", s0, "->", s1)

S2: 496 -> 445


In [26]:
# 10. Weather 1st rule W1: keep the routine hourly reports FM-15 (the solution proposed to the problem #12)
weather = spark.read.parquet(BRONZE + "/weather")
w0 = weather.count()
weather.groupBy("REPORT_TYPE").count().show()
weather = weather.where("REPORT_TYPE = 'FM-15'")
w1 = weather.count()
print("W1:", w0, "->", w1)

+-----------+-----+
|REPORT_TYPE|count|
+-----------+-----+
|        SOM|   12|
|      FM-15| 8771|
|        SOD|  365|
|      FM-16| 2617|
+-----------+-----+

W1: 11765 -> 8771


In [27]:
# 11. Weather 2nd W2: convert NOAA time to New York clock time (the solution proposed to problem #11)
# NOAA uses winter time all year while New York is 1 hour ahead in summer time:
# before 3 Nov 2024 01:00 and from 9 Mar 2025 02:00 because winter time is the same for NOAA time and New York Time
weather = weather.selectExpr("*", "DATE + interval 1 hour AS DATE_plus_1h")
weather = weather.withColumn("ny_time",
    when(col("DATE") < "2024-11-03 01:00:00", col("DATE_plus_1h"))
    .when(col("DATE") >= "2025-03-09 02:00:00", col("DATE_plus_1h"))
    .otherwise(col("DATE")))

# a report at 13:51 describes the hour 13:00 -> same hours as the ridership table
weather = weather.withColumn("hour_local", to_timestamp(date_format(col("ny_time"), "yyyy-MM-dd HH:00:00")))
weather.select("DATE", "ny_time", "hour_local").show(3)

+-------------------+-------------------+-------------------+
|               DATE|            ny_time|         hour_local|
+-------------------+-------------------+-------------------+
|2025-05-01 00:51:00|2025-05-01 01:51:00|2025-05-01 01:00:00|
|2025-05-01 01:51:00|2025-05-01 02:51:00|2025-05-01 02:00:00|
|2025-05-01 02:51:00|2025-05-01 03:51:00|2025-05-01 03:00:00|
+-------------------+-------------------+-------------------+
only showing top 3 rows



In [28]:
# 12. Weather 3rd rule W3: rain "T" (trace) that indicated a negiligible amount of rain of 0.1 mm which is negilible. For that, we convert it to 0 mm, so here e convert every column that has "T" as a text into a number ( The solution proposed for problem #13)
weather = weather.withColumn("precip_mm",
    when(col("HourlyPrecipitation") == "T", 0.0)
    .otherwise(col("HourlyPrecipitation").cast("double")))
weather = weather.withColumn("snow",
    when(col("HourlyPresentWeatherType").contains("SN"), 1).otherwise(0))
weather.describe("HourlyDryBulbTemperature", "precip_mm").show()

+-------+------------------------+-------------------+
|summary|HourlyDryBulbTemperature|          precip_mm|
+-------+------------------------+-------------------+
|  count|                    8747|               8391|
|   mean|       13.50093746427347|0.12705279466094618|
| stddev|        9.74439352438651| 0.9216220747047408|
|    min|                   -11.7|                0.0|
|    max|                    36.7|               32.5|
+-------+------------------------+-------------------+



In [29]:
# 13. Weather 4th rule W4: we gather into one row per New York hour, with a label
weather_hour = weather.groupBy("hour_local").agg(
    avg("HourlyDryBulbTemperature").alias("temp_c"),
    max("precip_mm").alias("precip_mm"),
    max("snow").alias("snow"),
)
weather_hour = weather_hour.where("hour_local >= '2024-07-01 00:00:00' and hour_local < '2025-07-01 00:00:00'")
weather_hour = weather_hour.withColumn("condition",
    when(col("snow") == 1, "snow")
    .when(col("precip_mm") >= 2.5, "heavy rain")
    .when(col("precip_mm") > 0, "light rain")
    .when(col("precip_mm") == 0, "dry"))
weather_hour.write.mode("overwrite").parquet(SILVER + "/weather_hour")
w2 = spark.read.parquet(SILVER + "/weather_hour").count()
print("W4:", w1, "->", w2)
weather_hour.groupBy("condition").count().show()

W4: 8771 -> 8704
+----------+-----+
| condition|count|
+----------+-----+
|      NULL|  317|
|       dry| 7753|
|light rain|  459|
|heavy rain|  110|
|      snow|   65|
+----------+-----+



In [ ]:
# 14. We create Cleaning report: rows before / after each step
report = spark.createDataFrame([
    ("ridership", "R1 keep the subway only", r0, r1),
    ("ridership", "R2 drop missing or negative values", r1, r2),
    ("ridership", "R3 station id text -> integer", r2, r3),
    ("ridership", "R4 drop the hour that does not exist (2025-03-09 02:00)", r3, r4),
    ("ridership", "R5 drop duplicate rows", r4, r5),
    ("ridership", "R6 one row per station and hour", r5, r6),
    ("stations", "S2 one row per station complex", s0, s1),
    ("weather", "W1 keep the hourly reports FM-15", w0, w1),
    ("weather", "W4 one row per New York hour", w1, w2),
], ["table", "step", "rows_before", "rows_after"])
report.show(truncate=False)
report.coalesce(1).write.mode("overwrite").option("header", True).csv(RESULTS + "/cleaning_report")

+---------+-------------------------------------------------------+-----------+----------+
|table    |step                                                   |rows_before|rows_after|
+---------+-------------------------------------------------------+-----------+----------+
|ridership|R1 keep the subway only                                |29375687   |29145131  |
|ridership|R2 drop missing or negative values                     |29145131   |29145131  |
|ridership|R3 station id text -> integer                          |29145131   |29145131  |
|ridership|R4 drop the hour that does not exist (2025-03-09 02:00)|29145131   |29143881  |
|ridership|R5 drop duplicate rows                                 |29143881   |29143881  |
|ridership|R6 one row per station and hour                        |29143881   |3620146   |
|stations |S2 one row per station complex                         |496        |445       |
|weather  |W1 keep the hourly reports FM-15                       |11765      |8771      |